# openEO catalog explorer (no network, no auth)

The openEO catalog has two layers: **collections** (CDSE openEO collections you can load directly) and **recipes** (named process graphs). `earthlens.openeo.Catalog()` parses the bundled `catalog/` directory once per process - no network, no credentials - so you can browse what is curated and inspect a recipe's graph before issuing a download.

```bash
pip install earthlens[openeo]
```

In [ ]:
from earthlens.openeo import Catalog

cat = Catalog()
print('curated collections:', sorted(cat.datasets))
print('curated recipes:    ', sorted(cat.recipes))
print('available collections (full index):', len(cat.available_collections))
print('available processes  (full index):', len(cat.available_processes))

## Inspect a collection

`get_collection()` returns a curated collection row. Print its `collection_id`, cadence, native resolution, and the default vs. full band list so you know exactly what `load()` would request.

In [ ]:
col = cat.get_collection('sentinel-2-l2a')
print('collection_id:', col.collection_id)
print('cadence:      ', col.cadence)
print('resolution:   ', col.resolution, 'm')
print('default bands:', col.default_bands)
print('all bands:    ', col.bands)

## Inspect a recipe's process graph

A recipe is a named process graph layered on top of a base collection. `get_recipe()` exposes its base collection, bands, output format, and the ordered graph steps the backend will execute.

In [ ]:
recipe = cat.get_recipe('sentinel-2-l2a-ndvi-monthly')
print('base collection:', recipe.base_collection)
print('bands:          ', recipe.bands)
print('output format:  ', recipe.output_format)
print('graph steps:')
for step in recipe.graph:
    name = next(iter(step))
    print(f'  - {name}: {step[name]}')

## Resolve any key to a uniform shape

`resolve()` normalises a collection or a recipe to one `ResolvedGraph` (the shape the backend builds from).

In [ ]:
for key in ['sentinel-1-grd', 'sentinel-5p-no2-monthly']:
    g = cat.resolve(key)
    print(
        f'{key}: collection_id={g.collection_id}, is_recipe={g.is_recipe}, '
        f'bands={g.bands}, steps={[next(iter(s)) for s in g.graph]}'
    )

An unknown key raises a `ValueError` with a closest-match hint:

In [ ]:
try:
    cat.resolve('sentinel2')
except ValueError as exc:
    print(exc)